In [2]:
from collections import defaultdict

In [4]:
import json


with open("data.json", "r") as f:
    data = json.load(f)

print("Total Records:", len(data))

Total Records: 59


In [5]:
def clean(record):
    record["carrier"] = record["carrier"] or "UNKNOWN"
    record["voiceQuality"] = record["voiceQuality"] if record["voiceQuality"] is not None else -1
    record["audioIssues"] = record["audioIssues"] or []
    record["environment"] = record["environment"] or "UNKNOWN"
    
    record["duration_sec"] = record["callDuration"] / 1000
    
    return record

data = [clean(r) for r in data]

In [6]:
from datetime import datetime

for r in data:
    r["time"] = datetime.fromtimestamp(r["timestamp"] / 1000)

In [7]:
qualities = [r["voiceQuality"] for r in data if r["voiceQuality"] != -1]
avg_quality = sum(qualities) / len(qualities) if qualities else 0

print("Average Quality:", avg_quality)

Average Quality: 4.475


In [8]:
drops = sum(1 for r in data if "CALL_DROPPED" in r["audioIssues"])
total = len(data)

print("Drop Rate:", drops / total)

Drop Rate: 0.03389830508474576


In [9]:
durations = defaultdict(list)

for r in data:
    durations[r["networkGeneration"]].append(r["duration_sec"])

for net, vals in durations.items():
    print(net, sum(vals)/len(vals))

WiFi 230.27376470588234
LTE 230.82872


In [10]:


carrier_quality = defaultdict(list)

for r in data:
    if r["voiceQuality"] != -1 and r["carrier"] != "UNKNOWN":
        carrier_quality[r["carrier"]].append(r["voiceQuality"])

for carrier, vals in carrier_quality.items():
    print(carrier, sum(vals)/len(vals))

AIRTEL 4.428571428571429
Vi India 5.0


In [11]:
env_quality = defaultdict(list)

for r in data:
    if r["voiceQuality"] != -1 and r["environment"] != "UNKNOWN":
        env_quality[r["environment"]].append(r["voiceQuality"])

for env, vals in env_quality.items():
    print(env, sum(vals)/len(vals))

INDOOR 4.46875
OUTDOOR 4.333333333333333


In [17]:

dropped_calls = [r for r in data if "CALL_DROPPED" in r["audioIssues"]]
for call in dropped_calls:
    print(f"Time: {call['time']}, Carrier: {call['carrier']}, Network: {call['networkGeneration']}, Environment: {call['environment']}, Duration: {call['duration_sec']}s, Quality: {call['voiceQuality']}")


Time: 2026-04-02 14:49:05.418000, Carrier: AIRTEL, Network: WiFi, Environment: INDOOR, Duration: 0.917s, Quality: 2
Time: 2026-04-02 12:42:11.846000, Carrier: UNKNOWN, Network: WiFi, Environment: INDOOR, Duration: 2.668s, Quality: 5
